In [1]:
! pip install confluent_kafka

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
! python -m pip install confluent-kafka

     ---------------------------------------- 4.1/4.1 MB 8.2 MB/s eta 0:00:00



[notice] A new release of pip available: 22.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup:

1. Running a Local Broker (Docker) (Kafka Raft --> KRaft)
    - Install Docker
    - create docker-compose.yaml (with the commands required to setup a broker)
    - Run the docker-compose.yaml: Open the terminal in the folder that has the yaml file and type: `docker compose up -d`
    - Verify: Our broker is now listening on localhost:9092.

2. Setting up a topic
3. Setting a producer
4. Producing some messages and sending them through the producer
5. Setting up a consumer and consuming the messages

## 1. Create a Kafka Topic
Before sending messages, we need a "folder" to store them. While many brokers create topics automatically, defining them explicitly gives us control over partitions and replication.

In [2]:
from confluent_kafka.admin import AdminClient, NewTopic

def create_topic(bootstrap_servers, topic_name, num_partitions, replication_factor):
    admin_client = AdminClient({'bootstrap.servers': bootstrap_servers})
    
    # Define topic: name, number of partitions, and replication factor
    topic = NewTopic(topic_name, num_partitions=num_partitions, replication_factor=replication_factor)
    
    fs = admin_client.create_topics([topic])

    for topic, f in fs.items():
        try:
            f.result()  # The result itself is None
            print(f"Topic '{topic}' created successfully.")
        except Exception as e:
            print(f"Failed to create topic '{topic}': {e}")

create_topic(bootstrap_servers = 'localhost:9092', topic_name = 'roy_topic1',
             num_partitions = 3, replication_factor=1)

Topic 'roy_topic1' created successfully.


To check our existing topics, we can use the `list_topics()` method from the `AdminClient`. This is essentially the programmatic version of running kafka-topics --list in your terminal.

In [3]:
def list_existing_topics(bootstrap_servers):
    # Initialize the AdminClient
    admin_client = AdminClient({'bootstrap.servers': bootstrap_servers})
    
    # Fetch cluster metadata
    metadata = admin_client.list_topics(timeout=10)
    
    print("Existing Kafka Topics:")
    print("-" * 30)
    
    # metadata.topics is a dict where keys are topic names
    for topic_name in metadata.topics:
        print(f" - {topic_name}")

# Usage
list_existing_topics(bootstrap_servers = 'localhost:9092')

Existing Kafka Topics:
------------------------------
 - roy_topic1


### 2. The Producer
- The Producer sends messages to the topic.
- It's best practice to use a "delivery report" callback to ensure the message actually arrived.

In [ ]:
from confluent_kafka import Producer

def delivery_report(err, msg):
    if err is not None:
        print(f'Message delivery failed: {err}')
    else:
        print(f'Message delivered to {msg.topic()} [{msg.partition()}]')

p = Producer({'bootstrap.servers': 'localhost:9092'})

data = ["Hello Kafka!", "Sending message 2", "Final signal"]

for val in data:
    # Trigger any available delivery report callbacks from previous produce() calls
    p.poll(0)
    p.produce('my_awesome_topic', val.encode('utf-8'), callback=delivery_report)

p.flush() # Wait for all messages to be delivered